# Projet 5 Quand un océan devient possible

**Question.** Pourquoi ne suffit-il pas de dire « il fait moins de 100 °C » ? **Ancrage :** document 3, § 16–19, diagramme d’état. Lecture et interpolation.

**Entrées.** state : T_K ; diag : P_Pa, ph2o_Pa ; params : water_mode, choisi « pure_demo » ou « mixed_screen ». La routine ne retire aucun stock de vapeur.

**Données de frontière.** P en bar : 1, 10, 60, 100 ; températures d’ébullition arrondies en °C : 100, 180, 275, 311. Le point à 60 bar reprend la lecture du cours ; les autres valeurs arrondies sont ajoutées pour l’exercice. Les interpolations sont approximatives. Pas d’extrapolation hors 1–100 bar. Point triple : 0,01 °C et 610 Pa ; point critique : 374 °C et 221 bar.

**Modèle minimal.** En mode pure_demo, interpoler la température d’ébullition selon P et classer liquide si 273,15 K < T < Teb ; vapeur au-dessus ; frontière si égalité ; glace sous 273,15 K. C’est une approximation de l’eau pure limitée au domaine annoncé. Hors domaine, renvoyer « indetermine ».

**Mode du moteur.** En mélange, fixer ocean_possible = True seulement si T > 273,15 K et si ph2o_Pa dépasse la pression de saturation interpolée à T entre 100 et 311 °C. Aux autres températures, renvoyer None et water_domain = « indetermine ». Le test signale une possibilité de condensation, pas la quantité d’eau liquide. L’altération n’utilise True que dans ce domaine ou une valeur imposée explicitement par le scénario tempéré.

**Sorties.** diag : water_domain et ocean_possible. Aucun rates. En scénario tempéré, le professeur peut fournir un substitut « océan présent » avant l’assemblage du modèle de mélange.

**Validation.** Eau pure à 200 °C et 100 bar : liquide ; à 200 °C et 1 bar : vapeur. Tester une entrée hors domaine. Deux mélanges de même P et T mais de ph2o différent ne doivent pas être automatiquement classés de la même façon.

**Rendu.** Routine eau.py, diagramme et comparaison des deux modes. **Prolongement :** ajouter une table de saturation à basse température, puis un réservoir océanique conservant h2o total lors des transferts.

Lire aussi le contrat commun et le complément eau du dossier collectif.


## Travailler dans le navigateur

Ouvrir ce fichier .ipynb dans https://notebook.basthon.fr ou Colab. Aucun fichier Python annexe ni installation locale ne sont requis. Exécuter les cellules de haut en bas avec Maj + Entrée. Compléter uniquement les zones **À VOUS**, puis relancer les cellules qui suivent. Télécharger le notebook à la fin et le déposer dans votre dossier OneDrive G05. Le notebook ne se sauvegarde pas automatiquement dans OneDrive.

Les fonctions de référence du banc d’essai sont fournies ; elles ne sont pas la routine de votre groupe. Les formules sont dans la fiche ci-dessus et le complément commun.

## À VOUS — raisonnement avant le code

Membres et rôles : …

Entrées, sorties et unités : …

Formule symbolique et hypothèses : …

Calcul manuel : données … ; résultat attendu …

Cas limite prévu : …

In [ ]:
from copy import deepcopy
from math import exp, pi, isfinite, isclose
from random import Random
import matplotlib.pyplot as plt
def base():
    s = {"time_yr": 0., "T_K": 290., "ice": 0.,
         "co2_mol": 7.49e16, "n2_mol": 1.39e20, "o2_mol": 3.74e19,
         "h2o_mol": 0., "mantle_c_mol": 1e20, "rock_c_mol": 0.,
         "fe_mol": 0., "oxide_fe_mol": 0., "population": 0.,
         "fossil_J": 0., "emitted_c_mol": 0., "diag": {}}
    p = {"g_m_s2": 9.81, "area_m2": 5.10e14, "G": 6.67e-11,
         "kB": 1.38e-23, "NA": 6.02e23, "h": 6.63e-34, "c": 3e8,
         "mass_kg": 5.97e24, "radius_m": 6.371e6, "T_exo_K": 1000.,
         "S_W_m2": 1361., "sigma": 5.67e-8, "pref_Pa": 42.,
         "tau_T_yr": 10., "tau_ice_yr": 20.,
         "water_mode": "mixed_screen", "ocean_prescribed": True,
         "volcanic_mol_yr": 0., "weather_mol_yr": 0.,
         "land_fraction": 1., "co2_ref_mol": 7.49e16,
         "oxygen_mol_yr": 0., "lambda_nm": 255.,
         "ozone_column_mol_m2": .10, "absorption_m2_mol": 100.,
         "need_J_person_yr": 1e9, "renewable_J_yr": 4e11,
         "fossil_capacity_J_yr": 3e11, "carbon_mol_J": 2e-6,
         "birth_rate_yr": 0., "death_rate_yr": 0., "deficit_death_rate_yr": 0.}
    return s, p
"""Un moteur deterministe : lire, additionner les flux, avancer, verifier."""
from copy import deepcopy
from math import isfinite

STOCKS = ["co2_mol", "n2_mol", "o2_mol", "h2o_mol", "mantle_c_mol",
          "rock_c_mol", "fe_mol", "oxide_fe_mol", "population", "fossil_J",
          "emitted_c_mol"]
VARIABLES = STOCKS + ["T_K", "ice"]

def verifier_etat(s):
    for cle in VARIABLES + ["time_yr"]:
        if not isfinite(s[cle]) or s[cle] < 0:
            raise ValueError(f"Valeur invalide : {cle} = {s[cle]}")
    if s["ice"] > 1 or s["T_K"] <= 0:
        raise ValueError("Fraction de glace ou temperature invalide")

def verifier_diagnostic(x):
    if isinstance(x, dict):
        for valeur in x.values():
            verifier_diagnostic(valeur)
    elif isinstance(x, (int, float)) and not isfinite(x):
        raise ValueError("Diagnostic non fini")
    elif not isinstance(x, (str, bool, int, float, type(None))):
        raise ValueError("Type de diagnostic invalide")

def evaluer(s, p, modules, dt):
    travail = deepcopy(s)
    travail["diag"] = {}
    taux = {cle: 0.0 for cle in VARIABLES}
    for module in modules:
        try:
            r = module(deepcopy(travail), deepcopy(p), dt)
            if set(r) != {"rates", "values", "diag"} or r["values"]:
                raise ValueError("Contrat de sortie incorrect")
            for cle, valeur in r["rates"].items():
                if cle not in taux or not isfinite(valeur):
                    raise ValueError(f"Taux invalide : {cle}")
                taux[cle] += valeur
            for cle, valeur in r["diag"].items():
                if cle in travail["diag"]:
                    raise ValueError(f"Diagnostic duplique : {cle}")
                verifier_diagnostic(valeur)
                travail["diag"][cle] = valeur
        except (ValueError, KeyError, TypeError) as erreur:
            raise ValueError(f"{module.__module__}, t={s['time_yr']}, dt={dt} : {erreur}") from erreur
    return travail, taux

def planet_evolution(initial, params, modules, steps, dt_yr):
    if not isfinite(dt_yr) or dt_yr <= 0:
        raise ValueError("Le pas doit etre positif et fini")
    if not isinstance(steps, int) or steps < 0:
        raise ValueError("Le nombre de pas doit etre entier et >= 0")
    s = deepcopy(initial)
    verifier_etat(s)
    histoire = []
    for numero in range(steps + 1):
        # On enregistre les diagnostics du MEME instant que les stocks.
        travail, taux = evaluer(s, params, modules, dt_yr)
        histoire.append(deepcopy(travail))
        if numero == steps:
            break
        suivant = deepcopy(travail)
        for cle in VARIABLES:
            suivant[cle] = s[cle] + dt_yr * taux[cle]
        suivant["time_yr"] += dt_yr
        verifier_etat(suivant)
        s = suivant
    return histoire

def simuler_isole(initial, params, routine, steps, dt):
    """Banc d'essai : diagnostics voisins prescrits et conservés.
    Ce banc ne remplace pas le moteur collectif, qui recalcule les voisins.
    """
    s=deepcopy(initial);h=[deepcopy(s)]
    for _ in range(steps):
        r=routine(deepcopy(s),deepcopy(params),dt)
        assert set(r)=={'rates','values','diag'} and not r['values']
        for cle,taux in r['rates'].items():
            s[cle]+=dt*taux
            assert isfinite(s[cle]) and s[cle]>=0,cle
        s['time_yr']+=dt
        if 'ice' in s:assert s['ice']<=1
        h.append(deepcopy(s))
    return h


In [ ]:
# Projet 5 : interpolation, eau pure ou possibilite de condensation.
# Table arrondie ajoutee au cours. Aucune masse d'eau n'est transferee.
TABLE = [(0.01, 610), (10, 1230), (20, 2340), (30, 4240),
         (50, 12350), (80, 47400), (100, 100000),
         (180, 1000000), (275, 6000000), (311, 10000000)]

def interpoler(x, points):
    for i in range(len(points) - 1):
        x0, y0 = points[i]
        x1, y1 = points[i + 1]
        if x0 <= x <= x1:
            return y0 + (y1 - y0) * (x - x0) / (x1 - x0)
    return None



## À VOUS — écrire la routine

Lire les entrées de la fiche, calculer la relation et traiter les cas limites. Retourner rates, values vide et diag avec les clés exactes du contrat.

Remplacer le `raise NotImplementedError` par votre code. Les tests indiquent les valeurs attendues ; ils ne donnent pas votre implémentation.

In [ ]:
def step(s, p, dt):
    # À VOUS : écrire ici le modèle donné dans la fiche.
    raise NotImplementedError("routine du groupe 5")


## Vérifier votre fonction

Avant de coder, la cellule affiche « À compléter ». Une fois la routine écrite, une assertion qui échoue signifie que le test doit être examiné. Noter **attendu / obtenu / conclusion**, même quand les assertions passent.

In [ ]:
try:
    s,p=base();p.update(water_mode='pure_demo',ocean_prescribed=False)
    s['T_K']=473.15;s['diag']={'P_Pa':1e7,'ph2o_Pa':1e7}
    assert step(s,p,1)['diag']['ocean_possible'] is True
    s['diag']['P_Pa']=1e5;assert step(s,p,1)['diag']['ocean_possible'] is False
    p['water_mode']='mixed_screen';s['T_K']=293.15;s['diag'].update(P_Pa=1e5,ph2o_Pa=1000)
    assert step(s,p,1)['diag']['ocean_possible'] is False
    s['diag']['ph2o_Pa']=3000;assert step(s,p,1)['diag']['ocean_possible'] is True
    s['T_K']=1000;assert step(s,p,1)['diag']['ocean_possible'] is None
    print("Vérifications numériques réussies. Ajouter votre propre essai ci-dessous.")
    s,p=base(); s0,p0=deepcopy(s),deepcopy(p)
    s['diag']={'P_Pa':1e5,'ph2o_Pa':1000}
    s0,p0=deepcopy(s),deepcopy(p);step(s,p,1)
    assert s==s0 and p==p0,"La routine a modifié les entrées"
except NotImplementedError as erreur:
    print("À compléter :", erreur)


## À VOUS — ajouter une vérification justifiée

Quel paramètre changez-vous ? Quel résultat prévoyez-vous et pourquoi ?

In [ ]:
# À VOUS : ajouter un test avec un résultat attendu et une justification.


## Première figure fournie comme point de départ

La cellule propose une expérience et des axes. Exécutez-la après avoir codé votre routine. Ensuite, modifiez un paramètre, comparez deux expériences et justifiez votre lecture. Ce graphique isolé n’est pas encore la planète collective.

In [ ]:
try:
    plt.figure(figsize=(7,4))
    s,p=base();p.update(water_mode='mixed_screen',ocean_prescribed=False)
    x=list(range(1,101));y=[]
    for T in x:
     q=deepcopy(s);q['T_K']=T+273.15;q['diag']={'P_Pa':1e5,'ph2o_Pa':1000}
     y.append(step(q,p,1)['diag']['saturation_Pa'])
    plt.plot(x,y);plt.xlabel('Température (°C)');plt.ylabel('Pression de saturation (Pa)')
    plt.title("Groupe 5 — eau")
    plt.grid(alpha=.25)
    plt.show()
except NotImplementedError as erreur:
    print("À compléter :", erreur)


## À VOUS — interprétation et raccordement

Prédiction avant la figure : …

Ce que montre la courbe : …

Explication par la formule : …

Limite de notre modèle : …

Voisin fournisseur et donnée de substitution à retirer : …

Sortie utilisée par un autre groupe : …

Contribution au graphe collectif et signe de la rétroaction : …

Livrer ce notebook complété, avec les résultats visibles et le nom des membres.

## Extension pluie et océans — deuxième étape guidée

Le projet 5 étudie d'abord le diagnostic de phase de sa fiche ; le complément suivant fournit toutes les lois de l'extension géologique. Elles sont pédagogiques. Il n'est pas demandé aux élèves de les deviner.

Entrées : T (K), fraction de glace i entre 0 et 1, vapeur V (kg), eau totale Wtot (kg), surface A (m²), pas dt (an). Paramètres : E0 = 5 × 10¹⁷ kg/an et tau = 0,02 an dans l'exemple. Ces valeurs se modifient avec le scénario.

Évaporation : E = E0 × (1 − i) × f(T), avec f(T) = exp[(T − 288)/30], borné entre 0,05 et 3. L'exponentielle est donnée et disponible par math.exp ; elle ne constitue pas un prérequis de cours. À 288 K et sans glace, E = E0 ; avec i = 1, E = 0.

Vapeur : dV/dt = E − V/tau. Sur un pas à E constant : Vnouveau = E tau + (Vancien − E tau) exp(−dt/tau). Le code limite la cible à Wtot. Précipitation moyenne : P = E − (Vnouveau − Vancien)/dt. Le débit instantané est V/tau. Distinguer ces deux valeurs dans les résultats.

Eau de surface = Wtot − V. Océan liquide = (1 − i)(Wtot − V). Eau glacée = i(Wtot − V). Bilan à vérifier : vapeur + océan + glace = Wtot. La couverture i est choisie comme proxy de la fraction massique gelée ; le modèle ne calcule pas les calottes ni la chaleur latente.

Pluie moyenne = P(1 − i)/A en mm/an ; neige = Pi/A en mm équivalent eau/an. 1 kg/m² d'eau correspond à 1 mm. À 288 K et i = 0, l'équilibre vapeur vaut E0 tau = 10¹⁶ kg. Pour A = 5,10 × 10¹⁴ m², la pluie vaut environ 980 mm/an. Ce résultat est une moyenne planétaire, pas la météo locale.

Altération du moteur géologique : k = (F0/Cref) × land × max[0, 1 + (T − 288)/50] × (P/E0) × (1 − i). Flux retiré instantané = kC. Avec F volcanique constant : Cnouveau = C exp(−kdt) + (F/k)[1 − exp(−kdt)] ; si k = 0, Cnouveau = C + Fdt. Le retrait est C + Fdt − Cnouveau ; la même quantité est ajoutée aux roches. Fdt est retiré du manteau. Les coefficients sont figés sur le pas, d'où la nécessité du test de convergence.

Figure : vapeur, océan et glace dans le temps ; pluie dans le temps ; comparer avec et sans action de la pluie sur l'altération. Tests : bilan eau, bilan du flux, état glacé et équilibre à 288 K. La boucle à expliquer : T augmente → évaporation et pluie augmentent → altération augmente → CO2 diminue → effet de serre diminue → T diminue.

Cette extension utilise des solutions analytiques et un équilibre climatique rapide. Elle ne se branche pas telle quelle comme taux dans le moteur annuel : ses stocks, son traitement du temps et ses unités sont décrits dans histoire_planete.py. L'assemblage des élèves commence avec le contrat annuel ; le professeur présente ensuite la réduction géologique.


In [ ]:
def hydrologie(T, ice, vapor, dt, p):
    # À VOUS : appliquer les relations fournies ci-dessus.
    # Retour : vapor_kg, ocean_kg, ice_kg, evap_kg_yr, precip_kg_yr,
    # rain_mm_yr et snow_mm_yr. Les débits sont moyennés sur le pas.
    raise NotImplementedError("cycle de l’eau")


In [ ]:
try:
    p={'water_kg':1.4e21,'rain_ref_kg_yr':5e17,'vapor_tau_yr':.02,'area_m2':5.10e14}
    r=hydrologie(288,0,1e16,1,p)
    assert isclose(r['vapor_kg'],1e16)
    assert isclose(r['rain_mm_yr'],5e17/5.10e14)
    assert isclose(r['vapor_kg']+r['ocean_kg']+r['ice_kg'],p['water_kg'])
    r=hydrologie(260,1,1e16,1,p)
    assert r['ocean_kg']==0 and r['rain_mm_yr']==0
    print('Bilans et cas limites du cycle de l’eau réussis.')
except NotImplementedError as erreur:
    print("À compléter :", erreur)


## Tester notre routine dans le projet commun

Le lien de classe Basthon charge aussi `classe_commune.py` depuis GitHub. Les autres groupes proviennent de la version commune affichée. Votre fonction locale remplace uniquement votre groupe, sans changer le dépôt. Si vous avez ouvert seulement le fichier .ipynb, charger aussi classe_commune.py comme module avec le bouton d’ouverture.

Une sauvegarde Basthon télécharge votre copie ; elle ne publie pas votre travail sur GitHub. Déposer le notebook et les résultats sur OneDrive. Après validation, le professeur met à jour la version commune ; les autres groupes la récupèrent au prochain chargement. Les références professeur restent identifiées tant qu’une routine élève n’a pas été validée.


In [ ]:
import json
try:
    from classe_commune import comparer_groupe, rapport_comparaison, VERSION_CLASSE
    print("Version commune :",VERSION_CLASSE)
    # Choix : chaude, gelee, deglaciation, societe, oxygenation.
    # Choisir un scénario qui sollicite votre mécanisme.
    resultat_classe=comparer_groupe(5,step,nom="societe")
    page=rapport_comparaison(resultat_classe)
    from IPython.display import HTML, display
    from base64 import b64encode
    from html import escape
    display(HTML('<iframe title="Notre contribution" style="width:100%;height:900px;border:0" sandbox="allow-scripts allow-modals" srcdoc="'+escape(page,quote=True)+'"></iframe>'))
    for contenu,nom in [(page,"G05_rapport.html"),(json.dumps(resultat_classe,ensure_ascii=False),"G05_resultats.json")]:
        encoded=b64encode(contenu.encode()).decode()
        display(HTML('<p><a download="'+nom+'" href="data:application/octet-stream;base64,'+encoded+'">Télécharger '+nom+'</a></p>'))
except ImportError:
    print("Ouvrir le lien Basthon de classe, ou charger classe_commune.py comme module.")
except NotImplementedError as erreur:
    print("Compléter d’abord votre routine :",erreur)


## Notre bilan collectif

Quelle version commune avons-nous utilisée ? …

Quelles routines viennent de nos camarades et lesquelles restent des références ? …

Quel effet produit notre routine dans ce scénario ? …

Déposer le notebook, le rapport et les résultats dans le dossier du groupe.